# Lab 03 — Logistic Regression — Telco Churn
**Classification Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Build a binary classifier with logistic regression
2. Encode categoricals and handle blank TotalCharges
3. Tune decision threshold for precision/recall trade-off
4. Read coefficients as log-odds for interpretability

## Datasets (this folder)
- `Telco-Customer-Churn.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-03-logistic-regression-churn/lab-03-logistic-regression-churn.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Telco-Customer-Churn.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-03-logistic-regression-churn"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-03-logistic-regression-churn/bundle/dataset.zip"
NEED = ["Telco-Customer-Churn.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Telecom: Predicting Customer Churn

> **Scenario:** You are a data scientist at a telecom company. The marketing team wants to identify customers likely to churn (leave) so they can proactively offer retention incentives. You have 7,043 customer records with 20 features and a binary `Churn` target. Your task is to build a logistic regression classifier, interpret its coefficients, and tune the decision threshold for business needs.
>
> **You will learn:** binary classification, categorical encoding, logistic regression coefficients as log-odds, confusion matrix, precision/recall, threshold tuning.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Binary classification | Predict one of two classes (churn / no churn) | `LogisticRegression()` |
| Log-odds | Log of the odds ratio; coefficients are additive on this scale | `model.coef_` |
| Sigmoid | Maps log-odds to probability (0–1) | `model.predict_proba()` |
| Decision threshold | Probability cutoff for classifying as positive | Default 0.5 |
| Confusion matrix | TP, FP, TN, FN counts | `confusion_matrix()` |
| Precision | Of predicted churners, how many actually churned | `precision_score()` |
| Recall | Of actual churners, how many did we catch | `recall_score()` |

---

## 1. Load and Explore the Data

The Telco Customer Churn dataset has 7,043 rows and 21 columns. The target is `Churn` (Yes/No). Features include demographics (gender, senior citizen, partner, dependents), account info (tenure, contract, billing), and services (internet, security, streaming).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"Telco-Customer-Churn.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df["Churn"].value_counts())
print(f"Churn rate: {(df['Churn'] == 'Yes').mean():.4f}")


**What to notice:**
- 7,043 rows, 21 columns (20 features + 1 target).
- The dataset is imbalanced: ~26.5% churn vs ~73.5% no-churn.
- `TotalCharges` is stored as object (string) because of blank values — needs cleaning.

> **Pitfall:** Class imbalance means accuracy alone is misleading. A model that always predicts "no churn" gets ~73.5% accuracy but is useless.

---

## 2. Clean and Encode the Data

`TotalCharges` contains blank strings for customers with zero tenure. We convert it to numeric, fill blanks with 0, then encode all categorical variables using one-hot encoding.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"Missing values: {X.isnull().sum().sum()}")


**What to notice:**
- After one-hot encoding, the feature matrix expands to ~30+ columns.
- The target is now binary: 1 = churn, 0 = no churn.
- No missing values remain after cleaning.

> **Pitfall:** `pd.to_numeric(errors="coerce")` turns blanks into NaN. Always fill them before modeling.

---

## 3. Fit Logistic Regression

Logistic regression models the log-odds of churn as a linear combination of features. The sigmoid function converts log-odds to probabilities. We fit on 80% of the data and evaluate on the held-out 20%.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


**What to notice:**
- Accuracy may look decent (~78–80%) but is inflated by class imbalance.
- Precision and recall give a clearer picture of model performance on the minority class.
- The confusion matrix shows the exact TP/FP/TN/FN breakdown.

> **Pitfall:** Use `stratify=y` in `train_test_split` to preserve the class ratio in both train and test sets.

---

## 4. Interpret Coefficients as Log-Odds

Logistic regression coefficients represent the change in log-odds of churn for a one-unit increase in the feature. A positive coefficient increases churn odds; a negative coefficient decreases them.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0]),
}).sort_values("coef", ascending=False)

print("Top 5 churn INCREASERS:")
print(coef_df.head(5).to_string(index=False))
print("\nTop 5 churn DECREASERS:")
print(coef_df.tail(5).to_string(index=False))


**What to notice:**
- `tenure` (negative coefficient) means longer-tenured customers are less likely to churn.
- `Contract_One year` or `Contract_Two year` (negative) means longer contracts reduce churn.
- `InternetService_Fiber optic` (positive) may indicate fiber customers churn more.
- Odds ratio > 1 means increased churn odds; < 1 means decreased.

> **Pitfall:** Coefficients are only comparable when features are on similar scales. Standardize features if you want to compare magnitudes directly.

---

## 5. Threshold Tuning

The default threshold of 0.5 may not be optimal for business needs. If the cost of missing a churner (lost revenue) is high, you might lower the threshold to increase recall at the expense of precision.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    results.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

target_recall = 0.80
valid = results_df[results_df["recall"] >= target_recall]
best_threshold = valid["threshold"].max()
print(f"\nHighest threshold with recall >= {target_recall}: {best_threshold}")


**What to notice:**
- Lowering the threshold increases recall (catch more churners) but decreases precision (more false alarms).
- The business must decide the acceptable trade-off.
- A threshold around 0.3–0.4 often achieves recall ≥ 0.80.

> **Pitfall:** Don't tune the threshold on the test set in production — use a validation set or cross-validation to avoid overfitting the threshold.

---

## Exercises (do these!)

### Exercise 1 — Churn Base Rate
What percentage of customers in the dataset churned?
*Expected: 26.54% (1,869 out of 7,043).*

**Follow-up:** Why is the base rate important for evaluating classifier performance? Check: base rate value.

### Exercise 2 — Precision and Recall at Threshold 0.5
Compute precision and recall using the default 0.5 threshold.
*Expected: Precision = 0.6531, Recall = 0.5588.*

**Follow-up:** Which metric matters more for a retention campaign — precision or recall? Check: both values.

### Exercise 3 — Threshold for Recall ≥ 0.80
Find the highest threshold that achieves recall of at least 0.80.
*Expected: 0.25 (precision = 0.4983, recall = 0.8075).*

**Follow-up:** What happens to precision at this threshold? Check: precision at the chosen threshold.

<details>
<summary>Hint</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# Exercise 1
print(f"Churn rate: {y.mean():.4f}")
print(f"Churn count: {y.sum()} / {len(y)}")

# Exercise 2
y_pred = (y_proba >= 0.5).astype(int)
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")

# Exercise 3
for t in np.arange(0.1, 0.9, 0.05):
    y_pred_t = (y_proba >= t).astype(int)
    r = recall_score(y_test, y_pred_t)
    if r >= 0.80:
        print(f"Threshold {t:.2f}: recall = {r:.4f}")
```

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# --- Solution 1 ---
churn_rate = y.mean()
churn_count = y.sum()
print(f"Churn rate: {churn_rate:.4f} ({churn_count}/{len(y)})")

# --- Solution 2 ---
y_pred_05 = (y_proba >= 0.5).astype(int)
prec = precision_score(y_test, y_pred_05)
rec = recall_score(y_test, y_pred_05)
print(f"Precision @ 0.5: {prec:.4f}")
print(f"Recall @ 0.5:    {rec:.4f}")

# --- Solution 3 ---
thresholds = np.arange(0.1, 0.9, 0.05)
best_t = None
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    if recall_score(y_test, y_pred_t) >= 0.80:
        best_t = t
print(f"Best threshold for recall >= 0.80: {best_t:.2f}")

# --- Follow-up 1 ---
assert 0.25 < churn_rate < 0.28, "Churn rate should be ~26.5%"
print(f"Base rate = {churn_rate:.4f} — accuracy must beat this to be useful.")

# --- Follow-up 2 ---
print(f"Precision = {prec:.4f}, Recall = {rec:.4f}")
print("For retention, recall matters more — catching churners is priority.")

# --- Follow-up 3 ---
y_pred_best = (y_proba >= best_t).astype(int)
prec_at_best = precision_score(y_test, y_pred_best)
rec_at_best = recall_score(y_test, y_pred_best)
assert rec_at_best >= 0.80, "Recall must be >= 0.80"
print(f"At threshold {best_t:.2f}: precision = {prec_at_best:.4f}, recall = {rec_at_best:.4f}")


### What to learn next
- Handling class imbalance with SMOTE or class weights
- ROC curves and AUC for threshold-independent evaluation
- Gradient boosting classifiers (XGBoost) for churn prediction

*Files in this folder: Telco-Customer-Churn.csv.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
